In [ ]:
# !pip install uv

In [ ]:
!uv pip install openai==2.31.0 pandas==2.3.3 openpyxl==3.1.5 arize==8.11.0 arize-otel==0.12.0 openinference-instrumentation-openai==0.1.44 openinference-instrumentation-openai-agents==1.4.1 openai-agents==0.13.6 python-dotenv==1.2.2

In [ ]:
# !git clone https://github.com/rk-yen/customer-support-collab.git

# Secrets (Before Starting)
Add the following variables in the Secrets section:
- `ARIZE_API_KEY`
- `ARIZE_SPACE_ID`
- `OPENAI_API_KEY`

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
pd.set_option('max_colwidth', None)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'workshop_helpers').exists():
    candidates = [path for path in REPO_ROOT.iterdir() if path.is_dir() and (path / 'workshop_helpers').exists()]
    if not candidates:
        raise FileNotFoundError('Could not find a repo folder containing workshop_helpers')
    REPO_ROOT = candidates[0]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workshop_helpers.backend import TOOLS, hydrate_backend_from_dataset
from workshop_helpers.data import DATASET
from workshop_helpers.experiments import (
    V2_ALL_SPECIALISTS_DATASET_NAME,
    V2_BILLING_DATASET_NAME,
    build_review_context_message,
    dispatch_specialist_response,
    ensure_arize_dataset,
    ensure_brand_voice_calibration_dataset,
    prepare_experiment_bundle,
    run_experiment,
    select_cases,
    select_cases_by_categories,
    select_cases_by_category,
    summarize_dataset,
    summarize_router_experiment_results,
)
from workshop_helpers.metrics import (
    ExactMatchEvaluator,
    RoutingAccuracyEvaluator,
    judge_brand_voice,
    pack_response_payload,
    score_routing_response,
)
from workshop_helpers.scenarios import parse_router_raw_response, run_router_raw, run_raw_llm


In [ ]:
from workshop_helpers.setup import setup_clients

env = setup_clients()
client = env['client']
ARIZE_SPACE_ID = env['arize_space_id']
ARIZE_API_KEY = env['arize_api_key']

# Workiva - AI Support Copilot Workshop

We will work through two systems:
1. `v1` router classification with a code-based exact-match metric.
2. `v2` routed draft-reply creation with three specialist behaviors:
   - `permissions`: single LLM call
   - `review_workflow`: LLM call with injected workflow context
   - `billing`: tool-using billing draft agent

The routing approach uses a raw LLM response for classification, then parses the returned text into a category for evaluation.
The router selects a support category, then the notebook dispatches to the matching specialist to draft a reply.


## Setup

The scenario library has 50 cases total.
Use `LIMIT_N_CASES` for a faster workshop run if needed.

In [ ]:
LIMIT_N_CASES = 50  # Set to an integer like 10 for a faster run. For full dataset, set to None.
EXPERIMENT_DATASET = select_cases(DATASET, limit_n=LIMIT_N_CASES)
library_summary = summarize_dataset(DATASET)
experiment_summary = summarize_dataset(EXPERIMENT_DATASET)
backend_snapshot = hydrate_backend_from_dataset(DATASET)

print(f"Scenario library: {library_summary['scenario_count']} total cases")
print(f"Experiment dataset: {experiment_summary['scenario_count']} cases")
print(f"Categories: {', '.join(experiment_summary['categories'])}")
print(f"Limit applied: {LIMIT_N_CASES}")
print()
print('Demo backend state:')
for label, value in backend_snapshot.items():
    print(f'  {label}: {value}')

pd.DataFrame(EXPERIMENT_DATASET)[['scenario_id', 'category']].head(10)

### Create The Arize Dataset

Create the dataset once during setup so you can inspect it in Arize before running experiments.

In [ ]:
ARIZE_DATASET_BUNDLE = ensure_arize_dataset(
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=EXPERIMENT_DATASET,
)
arize_client = ARIZE_DATASET_BUNDLE['client']
DATASET_ID = ARIZE_DATASET_BUNDLE['dataset_id']
ROUTER_DATASET_ID = DATASET_ID

print(f"Arize dataset: {ARIZE_DATASET_BUNDLE['dataset_name']}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Rows uploaded: {ARIZE_DATASET_BUNDLE['row_count']}")
print(f"Created in this run: {ARIZE_DATASET_BUNDLE['created']}")

display(ARIZE_DATASET_BUNDLE['dataframe'].head(10))


In [ ]:
PERMISSIONS_CASE = select_cases_by_category(DATASET, 'permissions')[0]
REVIEW_CASE = select_cases_by_category(DATASET, 'review_workflow')[0]
BILLING_CASE = select_cases_by_category(DATASET, 'billing')[0]
ESCALATION_CASE = select_cases_by_category(DATASET, 'escalation')[0]
ROUTING_CATEGORIES = sorted({case['category'] for case in DATASET})
print(f"The available routing categories: {ROUTING_CATEGORIES}")

# We are picking a billing case for the demo
ROUTER_DEMO_CASE = BILLING_CASE

V2_BILLING_DATASET = select_cases_by_category(DATASET, 'billing')
V2_ALL_SPECIALISTS_DATASET = select_cases_by_categories(DATASET, ['permissions', 'review_workflow', 'billing'])
print(f"Billing-only v2 cases: {len(V2_BILLING_DATASET)}")
print(f"All-specialist v2 cases: {len(V2_ALL_SPECIALISTS_DATASET)}")


---
## Part 1 - `v1` Routing Classification

We start with the simplest system: a router that maps each user message to one support category.

The loop is:
1. Run the router once.
2. Add a code-based exact-match eval.
3. Improve the prompt.
4. Re-run and compare the score.

In [ ]:
PROMPT_ROUTER_BASELINE = (
    'You are a support routing classifier. '
    'Classify a user request into one of the 4 categories: review_workflow, permissions, billing, escalation. '
    "Return ONLY a JSON object with the key 'category' and the appropriate category value."
)

router_baseline_raw = run_raw_llm(
    client,
    ROUTER_DEMO_CASE['user_input'],
    PROMPT_ROUTER_BASELINE,
    max_tokens=80,
)
router_baseline = parse_router_raw_response(router_baseline_raw, ROUTING_CATEGORIES)
router_payload_baseline = pack_response_payload(router_baseline['category'], metadata=router_baseline)

print('Router response:')
print(router_baseline_raw)

display(pd.DataFrame([
    {
        'scenario_id': ROUTER_DEMO_CASE['scenario_id'],
        'user_input': ROUTER_DEMO_CASE['user_input'],
        'expected_category': ROUTER_DEMO_CASE['category'],
        'predicted_category': router_baseline['category'],
        **score_routing_response(router_payload_baseline, ROUTER_DEMO_CASE['category']),
    }
]))


### Run The Router Experiment In Arize

Now that the dataset exists in Arize, we can run the router experiment and keep the traces and scores tied to that same dataset.
We start by running the baseline prompt across the dataset.


In [ ]:
def task_router_baseline(row: dict) -> str:
    route = run_router_raw(
        client,
        row['user_input'],
        PROMPT_ROUTER_BASELINE,
        ROUTING_CATEGORIES,
    )
    return route['category']

print('Running router baseline experiment without evaluators...')
run_router_baseline_traces = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-baseline-traces-only',
    task=task_router_baseline,
    evaluators=[],
)

print(f"Trace-only router experiment ID: {run_router_baseline_traces['experiment'].id}")
print(f"Rows evaluated: {len(run_router_baseline_traces['results_df'])}")


### Add The Evaluator And Re-Run

* We now have traces. But, how well is our system performing?
* To answer that question, we add an Evaluator. In this case, we are going to use the exact-match evaluator.
* This turns the full-dataset run from trace inspection into a measurable benchmark.


In [ ]:
router_eval = [RoutingAccuracyEvaluator(expected_field='category')]

print('Running router baseline experiment with exact-match evaluator...')
run_router_baseline = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-baseline',
    task=task_router_baseline,
    evaluators=router_eval,
)

router_baseline_summary = summarize_router_experiment_results(run_router_baseline['results_df'])
display(router_baseline_summary)

### Fix The Prompt And Re-Run The Same Experiment

Once we have the baseline score, we improve the router prompt and run the same Arize experiment flow again.
That lets us verify whether the score actually improved.

Couple of tools available for prompt optimization:
* From OpenAI: https://developers.openai.com/api/docs/guides/prompt-optimizer
* From Arize: https://arize.com/docs/ax/prompts/prompt-optimization
* From Arize: Using Alyx

### Let's try Alyx here.

Go to the experiment that we just ran. Click on **Ask Alyx** button. Chat with it to see if it can create an updated prompt for you.


In [ ]:
# From Alyx
PROMPT_ROUTER = '''You are a support routing classifier. Read the customer message and return exactly one category as raw JSON.

Categories:
- review_workflow: The customer is asking about workflow state or process, such as which review step a file is in, what is blocking progress, who still needs to approve, whether a reviewer or approver is required, whether a step can be skipped, what happens next, why the file moved backward, why a workflow exception happened, how to unblock a workflow, or why submit/publish is unavailable.
- permissions: The customer is asking about access rights, sharing, authorization, request path, who can grant access, why access was denied, why an access request is stuck, how to restore access, temporary access, cross-workspace access, contractor access, or how to bypass or speed up the access request path.
- billing: The customer is asking for billing operations help, such as explanation of an invoice, charge, refund, credit, fee, tax, proration, quote mismatch, payment issue, or what a billing response should say.
- escalation: The customer needs human ownership because either the issue is severe or it falls outside the supported workflow/permissions/billing taxonomy. This includes security or privacy exposure, legal or contract risk, abusive behavior, repeated severe unresolved failures, outage-like impact, explicit requests for a human or senior person, major deadline risk, product defects or technical investigation, data recovery or restore requests, sync/import/export/search/notification/integration failures, and commercial or account-management conversations such as competitor comparisons, discount requests, renewal negotiation, pricing strategy, procurement discussion, or contract-option review.

Tie-breakers:
1. Choose the customer's primary ask, not every topic mentioned.
2. If the message asks how to get access, who can grant access, why access was denied, or why an access request is pending or blocked, choose permissions even if workflow language is also present, unless the message says the access problem is causing severe business risk that needs immediate human attention, such as threatening a board meeting, filing, or comparable critical deadline.
3. If the message asks about workflow stage, approvers, blockers, required review steps, exceptions, publish/submit behavior, or how to unblock the process, choose review_workflow even if access is part of the surrounding context.
4. When access loss is mentioned only as one reason a workflow is stalled, and the customer is asking how to move the workflow forward, choose review_workflow.
5. Choose billing only for billing operations questions about explaining or handling a bill, invoice, charge, credit, refund, tax, fee, payment detail, or quote mismatch. Invoice-vs-quote questions stay in billing even if executives, urgency, or dissatisfaction are mentioned.
6. If pricing or invoice language is present but the real ask is competitor comparison, discount negotiation, commercial options, vendor evaluation, procurement, renewal strategy, or account-management discussion, choose escalation, not billing.
7. If the request is mainly about diagnosing, investigating, restoring, recovering, or escalating a product or platform problem rather than about workflow state, permissions, or billing operations, choose escalation.
8. Ordinary urgency alone does not force escalation, but severe business risk, explicit human-handoff requests, or unsupported issue types do. This includes access problems that put a board meeting, filing, or similarly critical event at immediate risk.

Return only this JSON shape: {"category": "review_workflow"}
Do not include explanations, markdown, or code fences.
'''

def task_router_improved(row: dict) -> str:
    route = run_router_raw(
        client,
        row['user_input'],
        PROMPT_ROUTER,
        ROUTING_CATEGORIES,
    )
    return route['category']

print('Running router improved experiment with exact-match evaluator...')
run_router_improved = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-improved',
    task=task_router_improved,
    evaluators=router_eval,
)

router_improved_summary = summarize_router_experiment_results(run_router_improved['results_df'])

router_comparison = pd.DataFrame([
    {
        'variant': 'router_baseline',
        'exact_match_accuracy': router_baseline_summary.loc[0, 'exact_match_accuracy'],
        'rows_evaluated': router_baseline_summary.loc[0, 'rows_evaluated'],
    },
    {
        'variant': 'router_improved_prompt',
        'exact_match_accuracy': router_improved_summary.loc[0, 'exact_match_accuracy'],
        'rows_evaluated': router_improved_summary.loc[0, 'rows_evaluated'],
    },
])

display(router_comparison)


---
## Part 2 - `v2` Billing Draft Replies

`v2` keeps the same router. And, for now, in the interest of time, we will be focusing only on billing cases.

We are not taking write actions yet. The billing specialist can look up account and invoice facts, but it should not apply credits, update invoices, or escalate billing cases on its own.

The other two specialists are included at the end of the notebook for you to run later:
- `permissions`: prompt-only
- `review_workflow`: prompt plus injected context

For now, the evaluated specialist is:
- `billing`: read-only tool-using draft agent


In [ ]:
PROMPT_PERMISSIONS = (
    'You are a permissions support specialist. '
    'Answer access questions clearly and concisely. '
    'Explain the right request path, who can grant access, and any policy limitation. '
    'Do not claim you changed permissions or approved access.'
)

PROMPT_REVIEW_WORKFLOW = (
    'You are a review-workflow support specialist. '
    'Use the provided workflow context to explain the current stage, blockers, and next best step. '
    'Do not invent missing approvers or workflow states.'
)

PROMPT_BILLING = (
    'You are a billing support specialist for an AI support copilot. '
    'The authenticated billing account ID for this session is: {authenticated_account_id}. '
    'Draft a reply only; do not claim you applied credits, updated invoices, reversed charges, or escalated a case. '
    'Use get_billing_account and get_invoice_details to verify facts before replying. '
    'Use read_billing_reference for policy guidance. '
    'If the account appears eligible for a credit or manual billing review, explain the next step and ask for confirmation before any action is taken.'
)

ESCALATION_RESPONSE_TEMPLATE = (
    "I'm sorry this requires urgent human attention. "
    "A strong draft should route {account_name} to a human specialist and clearly explain why."
)


In [ ]:
def run_v2_routed_demo(case: dict):
    route = run_router_raw(
        client,
        case['user_input'],
        PROMPT_ROUTER,
        ROUTING_CATEGORIES,
    )
    payload = dispatch_specialist_response(
        client=client,
        route_category=route['category'],
        case=case,
        prompt_permissions=PROMPT_PERMISSIONS,
        prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
        prompt_billing=PROMPT_BILLING,
        escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    )
    return route, json.loads(payload)

specialist_patterns = pd.DataFrame([
    {
        'category': 'billing',
        'pattern': 'read-only tool-using billing draft agent',
        'model_input_preview': BILLING_CASE['user_input'],
    },
])

print('Read-only billing tools available to the routed system:')
for tool in TOOLS:
    print(f'  - {tool.name}')

print('\v2 specialist:')
display(specialist_patterns)


In [ ]:
def summarize_v2_case(case: dict) -> dict:
    route, payload = run_v2_routed_demo(case)
    return {
        'scenario_id': case['scenario_id'],
        'expected_category': case['category'],
        'predicted_category': route['category'],
        'response_text': payload['response_text'],
    }

billing_demo = pd.DataFrame([
    summarize_v2_case(BILLING_CASE),
])

display(billing_demo[['scenario_id', 'expected_category', 'predicted_category', 'response_text']])


### Run The Billing `v2` Dataset In Arize

We will now use a new billing-only `v2` Arize dataset.
The router still sees all four route categories, but every dataset row is a billing test case so we can focus trace review on the tool-using billing specialist.


In [ ]:
PROMPT_BILLING = (
    'You are a billing support specialist for an AI support copilot. '
    'The authenticated billing account ID for this session is: {authenticated_account_id}. '
    'Draft a reply only; do not claim you applied credits, updated invoices, reversed charges, or escalated a case. '
    'Use get_billing_account and get_invoice_details to verify facts before replying. '
    'Use read_billing_reference for policy guidance. '
    'If the account appears eligible for a credit or manual billing review, explain the next step and ask for confirmation before any action is taken.'
)

v2_billing_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=V2_BILLING_DATASET,
    prompt_router=PROMPT_ROUTER,
    prompt_permissions=PROMPT_PERMISSIONS,
    prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
    prompt_billing=PROMPT_BILLING,
    escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    judge_prompts={},
    limit_n=LIMIT_N_CASES,
    dataset_name=V2_BILLING_DATASET_NAME,
    routing_categories=ROUTING_CATEGORIES,
)
arize_client = v2_billing_bundle['client']
V2_BILLING_DATASET_ID = v2_billing_bundle['dataset_id']
v2_billing_tasks = v2_billing_bundle['tasks']

print(f"Billing v2 Arize dataset: {v2_billing_bundle['dataset_name']}")
print(f"Dataset ID: {V2_BILLING_DATASET_ID}")
print(f"Rows uploaded: {v2_billing_bundle['row_count']}")
print(f"Created in this run: {v2_billing_bundle['created']}")

print('Running billing routed copilot experiment without evaluators...')
run_v2_billing_traces = run_experiment(
    arize_client=arize_client,
    dataset_id=V2_BILLING_DATASET_ID,
    name_prefix='v2-billing-routed-copilot-traces-only',
    task=v2_billing_tasks['task_v2_routed'],
    evaluators=[],
    concurrency=1,
)

print(f"Trace-only billing experiment ID: {run_v2_billing_traces['experiment'].id}")
print(f"Rows evaluated: {len(run_v2_billing_traces['results_df'])}")


### Trace Annotation And Error Analysis

At this point we have routed traces, but not quality scores yet.
This is the right moment to review examples and annotate failures by hand.

Right now, we will focus the annotation pass on two patterns:
- Brand voice: did the draft sound calm, professional, concise, and empathetic?
- Billing tool gaps: did the billing draft use the available account, invoice, and policy facts well enough to answer the user?


### Calibrate The Brand Voice Judge, Then Score Billing `v2`

Before using the brand voice judge on the billing routed-copilot experiment, create a small separate Arize dataset for judge calibration.
This keeps judge-prompt iteration separate from the main billing `v2` benchmark.

The scored billing `v2` evaluator set is intentionally simple:
- `brand_voice`: LLM judge

Billing tool issues stay in the trace annotation workflow for now. A typical fix is adding a missing read-only tool or improving an existing tool implementation, not adding an action metric.


In [ ]:
JUDGE_PROMPTS = {
    'brand_voice': (
        'You evaluate AI support draft responses for brand voice.\n\n'
        'Return exactly two lines in this format:\n'
        'LABEL: <Good|Acceptable|Poor>\n'
        'REASONING: <one sentence>\n\n'
        'GOOD: Calm, professional, concise, and empathetic. Does not claim unsupported write actions.\n'
        'ACCEPTABLE: Mostly professional but generic, awkward, slightly flat, or missing warmth.\n'
        'POOR: Blaming, abrupt, defensive, off-brand, or claims an action the draft system did not take.'
    ),
}

brand_voice_calibration_bundle = ensure_brand_voice_calibration_dataset(
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
)
BRAND_VOICE_DATASET_ID = brand_voice_calibration_bundle['dataset_id']

print(f"Brand voice calibration dataset: {brand_voice_calibration_bundle['dataset_name']}")
print(f"Dataset ID: {BRAND_VOICE_DATASET_ID}")
print(f"Rows uploaded: {brand_voice_calibration_bundle['row_count']}")
print(f"Created in this run: {brand_voice_calibration_bundle['created']}")
display(brand_voice_calibration_bundle['dataframe'])

def task_brand_voice_calibration(row: dict) -> str:
    label, reasoning = judge_brand_voice(
        client,
        pack_response_payload(row['response_text']),
        judge_prompts=JUDGE_PROMPTS,
    )
    return pack_response_payload(label, metadata={'reasoning': reasoning})

print('Running brand voice judge calibration experiment...')
run_brand_voice_calibration = run_experiment(
    arize_client=brand_voice_calibration_bundle['client'],
    dataset_id=BRAND_VOICE_DATASET_ID,
    name_prefix='brand-voice-judge-calibration',
    task=task_brand_voice_calibration,
    evaluators=[ExactMatchEvaluator(expected_field='expected_label')],
)

display(summarize_router_experiment_results(run_brand_voice_calibration['results_df']))


In [ ]:
# TODO: To calibrate the LLM judge with our human judges, we will be running the similar prompt optimization flow like we did for the router.
# Once we complete the above step, the users will be populating this cell with a newer prompt for the brand voice judge.

### Now run v2 dataset with evaluator

In [ ]:
v2_billing_scored_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=V2_BILLING_DATASET,
    prompt_router=PROMPT_ROUTER,
    prompt_permissions=PROMPT_PERMISSIONS,
    prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
    prompt_billing=PROMPT_BILLING,
    escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    judge_prompts=JUDGE_PROMPTS,
    limit_n=LIMIT_N_CASES,
    arize_client=arize_client,
    dataset_id=V2_BILLING_DATASET_ID,
    dataset_name=V2_BILLING_DATASET_NAME,
    routing_categories=ROUTING_CATEGORIES,
)
v2_billing_tasks = v2_billing_scored_bundle['tasks']

print('Running billing routed copilot experiment with evaluators...')
run_v2_billing = run_experiment(
    arize_client=arize_client,
    dataset_id=V2_BILLING_DATASET_ID,
    name_prefix='v2-billing-routed-copilot',
    task=v2_billing_tasks['task_v2_routed'],
    evaluators=v2_billing_scored_bundle['build_evaluators']('v2_routed'),
    concurrency=1,
)

print(f"Scored billing experiment ID: {run_v2_billing['experiment'].id}")
print(f"Rows evaluated: {len(run_v2_billing['results_df'])}")

---
## Optional: Run All Three Specialists Later

If there is time after the workshop, participants can enable the prompt-only permissions specialist and the context-aware review-workflow specialist.
This path creates a separate `v2` Arize dataset for all three specialists so the billing-only benchmark remains clean.


In [ ]:
optional_specialist_patterns = pd.DataFrame([
    {
        'category': 'permissions',
        'pattern': 'single LLM call',
        'model_input_preview': PERMISSIONS_CASE['user_input'],
    },
    {
        'category': 'review_workflow',
        'pattern': 'LLM call with injected workflow context',
        'model_input_preview': build_review_context_message(REVIEW_CASE['user_input'], REVIEW_CASE['source_data']),
    },
    {
        'category': 'billing',
        'pattern': 'read-only tool-using billing draft agent',
        'model_input_preview': BILLING_CASE['user_input'],
    },
])

display(optional_specialist_patterns)

v2_all_specialists_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=V2_ALL_SPECIALISTS_DATASET,
    prompt_router=PROMPT_ROUTER,
    prompt_permissions=PROMPT_PERMISSIONS,
    prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
    prompt_billing=PROMPT_BILLING,
    escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    judge_prompts=JUDGE_PROMPTS,
    limit_n=LIMIT_N_CASES,
    dataset_name=V2_ALL_SPECIALISTS_DATASET_NAME,
    routing_categories=ROUTING_CATEGORIES,
)
V2_ALL_SPECIALISTS_DATASET_ID = v2_all_specialists_bundle['dataset_id']
v2_all_specialists_tasks = v2_all_specialists_bundle['tasks']

print(f"All-specialists v2 Arize dataset: {v2_all_specialists_bundle['dataset_name']}")
print(f"Dataset ID: {V2_ALL_SPECIALISTS_DATASET_ID}")
print(f"Rows uploaded: {v2_all_specialists_bundle['row_count']}")
print(f"Created in this run: {v2_all_specialists_bundle['created']}")

run_v2_all_specialists = run_experiment(
    arize_client=v2_all_specialists_bundle['client'],
    dataset_id=V2_ALL_SPECIALISTS_DATASET_ID,
    name_prefix='v2-all-specialists-routed-copilot',
    task=v2_all_specialists_tasks['task_v2_routed'],
    evaluators=v2_all_specialists_bundle['build_evaluators']('v2_routed'),
    concurrency=1,
)

print(f"Scored all-specialists experiment ID: {run_v2_all_specialists['experiment'].id}")
print(f"Rows evaluated: {len(run_v2_all_specialists['results_df'])}")
